# Multi-Agent Orchestration for RAG Systems
# Step 1. RAG Baseline Reproduction

*Asignee: Alla* - *Review:*

This notebook reproduces baseline retrieval results for https://github.com/Trista1208/advanced_genAI.git (23.12.2025)

In this project, we reproduce the multilingual RAG baseline by reusing the provided retriever artifacts from the previous cohort and re-evaluating them in a single, unified notebook pipeline. The reported reproduction uses the `full_corpus` setup. We load BM25, Dense, and GraphRAG resources for the full corpus, then construct Hybrid and Re-ranking on top of the same retrieved candidates using consistent fusion and reranking logic. We recompute all metrics ourselves from the benchmark QA set and qrels with one shared evaluator, reporting Precision@k, Recall@k, and MRR identically for every method. This setup ensures reproducibility in our environment and keeps comparisons fair across methods on the realistic full-corpus setting.


In [1]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers pytrec_eval


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 58.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.

In [2]:
import os
import re
import json
import time
import pickle
import random
import pathlib
import importlib.util
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')


/tmp/ipykernel_699/420715519.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Setup complete.


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1.1 Setup

You need a hugging face token, https://huggingface.co/settings/tokens, add `HF_TOKEN` to Colab Secrets

To have all the necessary data in the right place you need to create a folder Adv_GenAI in your Google Drive and copy folders `benchmark` and `storage` **(and its content).** The respective paths are `/content/drive/MyDrive/Adv_GenAI/benchmark`
and `/content/drive/MyDrive/Adv_GenAI/storage` Copying took me around 20 minutes.


The reported project workflow uses the full corpus. Older subsample-compatible paths are kept only for artifact compatibility, not for the reported Step 1 results.

`EVAL_SCOPE = 'full_corpus'`




In [4]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

# Reported project scope: full_corpus
EVAL_SCOPE = 'full_corpus'
assert EVAL_SCOPE == 'full_corpus'
print('EVAL_SCOPE =', EVAL_SCOPE)

PATH_QA = PROJECT_ROOT / 'benchmark/benchmark_qa_bilingual.json'
PATH_QRELS_FIXED = PROJECT_ROOT / 'benchmark/score/fixed_size'

if EVAL_SCOPE == 'subsample':
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
    if not PATH_BM25_PICKLE.exists():
        PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval/fixed_size_chunk/bm25_retriever.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'
else:
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/full_corpus/retrieval/fixed_size_chunk/bm25_retriever_full.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/full_corpus/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/full_corpus/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/full_corpus/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

OUT_DIR = PROJECT_ROOT / f'results/baseline_repro_colab_{EVAL_SCOPE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    PATH_BM25_PICKLE, PATH_QA, PATH_QRELS_FIXED,
    PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required artifacts found.')


PROJECT_ROOT = /content/drive/MyDrive/Adv_GenAI
EVAL_SCOPE = full_corpus
All required artifacts found.


## 1.2 Evaluation Tools: BM25
**We start with preparing our own tools.** By adding small wrappers, we ensure that each method can be called in the same way and evaluated with the same metric pipeline, which keeps the baseline comparison consistent and technically fair.

Next cell prepares a compatibility **layer for BM25** so that pickled retrievers from different stages of the project can be used in one reproducible evaluation pipeline. In our setup, legacy and full-corpus artifacts are not always serialized with the same internal structure, so we load the original object and wrap it with a unified adapter that exposes a consistent search(query, top_k) interface. This avoids format-specific failures and ensures that BM25 can be evaluated with exactly the same downstream metric code as Dense, GraphRAG, Hybrid, and Re-ranking methods.

In [5]:
# Robust BM25 loader for legacy and full-corpus pickle formats
class BilingualBM25:
    """Compatibility class for notebook pickles."""

    def _rank_lang(self, q: str, lang: str, k: int):
        # Legacy object: self.bm25 + self.docs_by_lang
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def _get_docs_with_scores(self, ret, qq, top_k):
        # Full-corpus-style object: self.retrievers
        if hasattr(ret, 'get_relevant_documents_with_scores'):
            try:
                return ret.get_relevant_documents_with_scores(qq, k=top_k)
            except Exception:
                pass

        if hasattr(ret, 'vectorizer') and hasattr(ret, 'docs'):
            try:
                toks = qq.lower().split()
                scores = ret.vectorizer.get_scores(toks)
                ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
                return [(ret.docs[idx], float(score)) for idx, score in ranked]
            except Exception:
                pass

        if hasattr(ret, 'invoke'):
            try:
                old_k = getattr(ret, 'k', None)
                if old_k is not None:
                    ret.k = top_k
                docs = ret.invoke(qq)
                if old_k is not None:
                    ret.k = old_k
                return [(d, d.metadata.get('score', 0.0)) for d in docs[:top_k]]
            except Exception:
                pass

        return []

    def search(self, query: str, top_k: int = 100):
        # Route to correct behavior based on available attributes
        if hasattr(self, 'bm25') and hasattr(self, 'docs_by_lang'):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)
            for lang in ('en', 'de'):
                q_lang = translator.translate(query, lang) if translator and lang != src else query
                bag.extend(self._rank_lang(q_lang, lang, top_k))

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d
            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        if hasattr(self, 'retrievers') and isinstance(self.retrievers, dict):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)

            for lang, ret in self.retrievers.items():
                qq = translator.translate(query, lang) if translator and lang != src else query
                docs_with_scores = self._get_docs_with_scores(ret, qq, top_k)
                for doc, score in docs_with_scores:
                    doc.metadata['bm25_score'] = float(score)
                    bag.append(doc)

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid is None:
                    continue
                if uid not in best or d.metadata.get('bm25_score', -1e9) > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d

            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        raise AttributeError('Unsupported BilingualBM25 object format.')

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        if hasattr(self, 'base'):
            expanded = self._expand_query(query, self.base)
            return self.base.search(expanded, top_k)
        raise AttributeError('QEBM25 object missing base retriever.')

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_raw = pickle.load(f)

class BM25RetrieverAdapter:
    def __init__(self, obj):
        self.obj = obj

    def search(self, query: str, top_k: int = 100):
        # Primary path
        if hasattr(self.obj, 'search'):
            try:
                return self.obj.search(query, top_k=top_k)
            except TypeError:
                return self.obj.search(query, k=top_k)

        # LangChain retriever fallback
        if hasattr(self.obj, 'invoke'):
            old_k = getattr(self.obj, 'k', None)
            if old_k is not None:
                self.obj.k = top_k
            docs = self.obj.invoke(query)
            if old_k is not None:
                self.obj.k = old_k
            for rank, d in enumerate(docs, start=1):
                if hasattr(d, 'metadata'):
                    d.metadata.setdefault('bm25_score', float(top_k - rank))
            return docs[:top_k]

        raise AttributeError(f'Unsupported BM25 object type: {type(self.obj)}')

bm25_retriever = BM25RetrieverAdapter(bm25_raw)
print('BM25 loaded:', type(bm25_raw), '-> adapter ready')


BM25 loaded: <class '__main__.BilingualBM25'> -> adapter ready


## 1.3 Evaluation Tools: Dense Retriever
The next cell initializes the **Dense retriever** by loading the multilingual E5 embedding model and attaching it to the persisted Chroma index for the selected evaluation scope. Queries are formatted with the query: prefix expected by E5, and retrieved vector distances are converted into similarity scores for consistent ranking behavior. This gives us a semantic retrieval baseline that is directly comparable to BM25 in the shared evaluation framework.

In [6]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/tmp/ipykernel_699/1020676855.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)


Dense retriever ready.


## 1.4 Evaluation Tools: GraphRAG retriever
This cell defines and initializes the **GraphRAG retriever** by loading precomputed community embeddings, the community-to-chunk mapping, and the chunk corpus. At query time, it first selects the most relevant graph communities, then scores candidate chunks within those communities using embedding similarity, and returns the top-ranked documents with normalized GraphRAG scores. This provides a graph-informed retrieval baseline that complements lexical and dense retrieval in the final comparison.

In [7]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)

        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = 'C1', k_comms: int = 24, top_k: int = 100):
        L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100):
        return self.retrieve(query=query, level='C1', k_comms=24, top_k=top_k)

graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GraphRAG retriever ready.


## 1.5 Evaluation Tools: Hybrid Retrieval and Reranking
This cell defines the combined retrieval baselines used after the individual retrievers are loaded. Hybrid retrieval fuses BM25, Dense, and GraphRAG candidate lists with weighted reciprocal rank fusion to produce a single ranked output, while the re-ranking variant applies an additional overlap-based reranking step on top of fused candidates to improve top-ranked relevance. This allows us to evaluate both plain fusion and post-fusion refinement within the same framework.

In [8]:
# Hybrid + Re-ranking
def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    weights = weights or {'bm25': 1.2, 'dense': 1.0, 'graph': 0.6}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

class HybridRetriever:
    def __init__(self, bm25, dense, graph, rerank=False):
        self.bm25 = bm25
        self.dense = dense
        self.graph = graph
        self.rerank = rerank

    def search(self, query: str, top_k: int = 100):
        pre_k = max(30, top_k)
        bm = _safe_unique(self.bm25.search(query, top_k=pre_k))
        de = _safe_unique(self.dense.search(query, top_k=pre_k))
        gr = _safe_unique(self.graph.search(query, top_k=pre_k))

        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr})
        if self.rerank:
            fused = _overlap_rerank(fused[:max(50, top_k)], query, top_k=max(50, top_k))
        return fused[:top_k]

hybrid_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=False)
rerank_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=True)
print('Hybrid and ReRank retrievers ready.')


Hybrid and ReRank retrievers ready.


## 1.6 Baseline Evaluation (Benchmark Metrics)
This cell runs the shared evaluation pipeline for all baseline methods using the same QA set and qrels. For each query, it collects ranked document IDs, computes Precision@k, Recall@k, and reciprocal rank, and then aggregates results into per-query and per-method summary tables. Using one evaluation function for all methods ensures the reported baseline comparison is consistent and directly comparable.

In [9]:
# Evaluation
K_VALUES = [1, 3, 5, 10]
TOP_N = 100

def load_qrels(folder: pathlib.Path, threshold: float = 0.5):
    qrels = defaultdict(set)
    for fp in sorted(folder.glob('*.json')):
        did = fp.stem
        payload = json.loads(fp.read_text(encoding='utf-8'))
        for qid, rel in payload.items():
            if float(rel.get('relevance_score', 0.0)) >= threshold:
                qrels[str(qid)].add(did)
    return qrels

def normalize_docs(docs, top_n=100):
    out, seen = [], set()
    for d in docs:
        did = _uid(d)
        if did is None or did in seen:
            continue
        out.append(str(did))
        seen.add(did)
        if len(out) >= top_n:
            break
    return out

def precision_at_k(ranked, relevant, k):
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / max(k, 1)

def recall_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / len(relevant)

def reciprocal_rank(ranked, relevant):
    for i, x in enumerate(ranked, start=1):
        if x in relevant:
            return 1.0 / i
    return 0.0

qa_data = json.loads(PATH_QA.read_text(encoding='utf-8'))
qrels = load_qrels(PATH_QRELS_FIXED)

methods = {
    'BM25': bm25_retriever,
    'Dense': dense_retriever,
    'GraphRAG': graph_retriever,
    'Hybrid': hybrid_retriever,
    'ReRank': rerank_retriever,
}

runs = {}
for name, retriever in methods.items():
    run = {}
    for q in tqdm(qa_data, desc=f'Running {name}'):
        qid = str(q['id'])
        docs = retriever.search(q['question'], top_k=TOP_N)
        run[qid] = normalize_docs(docs, top_n=TOP_N)
    runs[name] = run

per_query_rows = []
summary_rows = []

for name, run in runs.items():
    qids = sorted(set(run.keys()) & set(qrels.keys()))
    mrr_vals = []
    p_vals = {k: [] for k in K_VALUES}
    r_vals = {k: [] for k in K_VALUES}

    for qid in qids:
        ranked = run[qid]
        rel = qrels[qid]
        row = {'method': name, 'qid': qid}

        rr = reciprocal_rank(ranked, rel)
        row['MRR'] = rr
        mrr_vals.append(rr)

        for k in K_VALUES:
            pk = precision_at_k(ranked, rel, k)
            rk = recall_at_k(ranked, rel, k)
            row[f'Precision@{k}'] = pk
            row[f'Recall@{k}'] = rk
            p_vals[k].append(pk)
            r_vals[k].append(rk)

        per_query_rows.append(row)

    summary = {'method': name, 'queries_evaluated': len(qids), 'MRR': float(np.mean(mrr_vals) if mrr_vals else 0.0)}
    for k in K_VALUES:
        summary[f'Precision@{k}'] = float(np.mean(p_vals[k]) if p_vals[k] else 0.0)
        summary[f'Recall@{k}'] = float(np.mean(r_vals[k]) if r_vals[k] else 0.0)
    summary_rows.append(summary)

per_query_df = pd.DataFrame(per_query_rows)
summary_df = pd.DataFrame(summary_rows).sort_values('MRR', ascending=False).reset_index(drop=True)

summary_df


Running BM25:   0%|          | 0/25 [00:00<?, ?it/s]

Running Dense:   0%|          | 0/25 [00:00<?, ?it/s]

Running GraphRAG:   0%|          | 0/25 [00:00<?, ?it/s]

Running Hybrid:   0%|          | 0/25 [00:00<?, ?it/s]

Running ReRank:   0%|          | 0/25 [00:00<?, ?it/s]

,method,queries_evaluated,MRR,Precision@1,Recall@1,Precision@3,Recall@3,Precision@5,Recall@5,Precision@10,Recall@10
0,GraphRAG,24,0.232573,0.083333,0.000687,0.097222,0.003166,0.116667,0.005892,0.116667,0.052857
1,ReRank,24,0.222952,0.041667,0.000196,0.138889,0.004480,0.125000,0.006323,0.100000,0.010619
2,Hybrid,24,0.202154,0.000000,0.000000,0.097222,0.003699,0.125000,0.028352,0.095833,0.031149
3,Dense,24,0.165547,0.041667,0.000147,0.069444,0.008953,0.058333,0.010095,0.066667,0.034097
4,BM25,24,0.151296,0.041667,0.001016,0.055556,0.001681,0.091667,0.005506,0.091667,0.011165


On the full-corpus setup, GraphRAG achieved the best overall ranking quality with the highest MRR (0.233), indicating that graph-guided retrieval was most effective at placing relevant evidence early in the ranked list. ReRank and Hybrid followed closely, with ReRank slightly outperforming Hybrid on MRR, which suggests that post-fusion refinement can improve early precision in some cases. Dense and BM25 showed lower MRR, indicating weaker top-rank relevance when used alone in this setting. Overall, the results suggest that full-corpus retrieval benefits from multi-source or graph-aware methods, while single-retriever baselines remain useful reference points but are less competitive at early-rank retrieval quality.

In [10]:
# Save deliverables
summary_path = OUT_DIR / 'metrics_summary.csv'
per_query_path = OUT_DIR / 'metrics_per_query.csv'
runs_path = OUT_DIR / 'runs.json'

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)
runs_path.write_text(json.dumps(runs, indent=2), encoding='utf-8')

print('Saved:')
print('-', summary_path)
print('-', per_query_path)
print('-', runs_path)


Saved:
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_summary.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_per_query.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/runs.json


## 1.7 Orchestration Evaluation (Reference Baseline Selection)

This section evaluates multi-retriever orchestration strategies separately from baseline methods:
- Voting
- Waterfall
- Confidence

These are reported in a separate table so baseline and orchestration analyses are not mixed.


In [11]:
# Orchestration retrievers
def _token_set(text: str):
    text = (text or '').lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(t for t in text.split() if t)

def _top_overlap(query: str, docs):
    if not docs:
        return 0.0
    q_terms = _token_set(query)
    top_text = docs[0].metadata.get('original_text') or docs[0].page_content or ''
    d_terms = _token_set(top_text)
    return len(q_terms & d_terms) / max(len(q_terms), 1)

class VotingRetriever:
    name = 'Voting'
    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights={'bm25': 1.2, 'dense': 1.0, 'graph': 0.6})
        return fused[:top_k]

class WaterfallRetriever:
    name = 'Waterfall'
    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': []}, weights={'bm25': 1.2, 'dense': 1.0, 'graph': 0.0})
        docs = fused[:top_k]

        # Fallback: if lexical overlap is too low, include GraphRAG
        if _top_overlap(query, docs) < 0.05:
            gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
            fused2 = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights={'bm25': 1.1, 'dense': 1.0, 'graph': 0.7})
            return fused2[:top_k]

        return docs

class ConfidenceRetriever:
    name = 'Confidence'

    @staticmethod
    def _route_weights(query: str):
        q = query.lower().strip()
        factoid = any(q.startswith(w) for w in ['who', 'when', 'where']) or any(ch.isdigit() for ch in q)
        semantic = any(w in q for w in ['why', 'how', 'explain', 'difference', 'impact'])

        if factoid:
            return {'bm25': 1.4, 'dense': 0.9, 'graph': 0.5}
        if semantic:
            return {'bm25': 0.9, 'dense': 1.3, 'graph': 0.6}
        return {'bm25': 1.0, 'dense': 1.1, 'graph': 0.5}

    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
        weights = self._route_weights(query)
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights=weights)
        return fused[:top_k]

orchestration_methods = {
    'Waterfall': WaterfallRetriever(),
    'Voting': VotingRetriever(),
    'Confidence': ConfidenceRetriever(),
}

print('Orchestration methods ready:', list(orchestration_methods.keys()))


Orchestration methods ready: ['Waterfall', 'Voting', 'Confidence']


In [12]:
# Evaluate orchestration methods with the same shared evaluator
orch_runs = {}
for name, retriever in orchestration_methods.items():
    run = {}
    for q in tqdm(qa_data, desc=f'Running {name}'):
        qid = str(q['id'])
        docs = retriever.search(q['question'], top_k=TOP_N)
        run[qid] = normalize_docs(docs, top_n=TOP_N)
    orch_runs[name] = run

orch_rows = []
for name, run in orch_runs.items():
    qids = sorted(set(run.keys()) & set(qrels.keys()))
    summary = {'method': name, 'queries_evaluated': len(qids)}

    mrr_vals = []
    p_vals = {k: [] for k in K_VALUES}
    r_vals = {k: [] for k in K_VALUES}

    for qid in qids:
        ranked = run[qid]
        rel = qrels[qid]
        mrr_vals.append(reciprocal_rank(ranked, rel))
        for k in K_VALUES:
            p_vals[k].append(precision_at_k(ranked, rel, k))
            r_vals[k].append(recall_at_k(ranked, rel, k))

    summary['MRR'] = float(np.mean(mrr_vals) if mrr_vals else 0.0)
    for k in K_VALUES:
        summary[f'Precision@{k}'] = float(np.mean(p_vals[k]) if p_vals[k] else 0.0)
        summary[f'Recall@{k}'] = float(np.mean(r_vals[k]) if r_vals[k] else 0.0)

    orch_rows.append(summary)

orch_summary_df = pd.DataFrame(orch_rows).sort_values('MRR', ascending=False).reset_index(drop=True)
orch_summary_df


Running Waterfall:   0%|          | 0/25 [00:00<?, ?it/s]

Running Voting:   0%|          | 0/25 [00:00<?, ?it/s]

Running Confidence:   0%|          | 0/25 [00:00<?, ?it/s]

,method,queries_evaluated,MRR,Precision@1,Recall@1,Precision@3,Recall@3,Precision@5,Recall@5,Precision@10,Recall@10
0,Confidence,24,0.208841,0.000000,0.000000,0.111111,0.003891,0.133333,0.028380,0.095833,0.031112
1,Waterfall,24,0.208303,0.041667,0.001344,0.138889,0.005348,0.100000,0.006052,0.070833,0.029732
2,Voting,24,0.202154,0.000000,0.000000,0.097222,0.003699,0.125000,0.028352,0.095833,0.031149


For orchestration on the full corpus, Confidence achieved the best MRR (0.209), with Waterfall very close (0.208) and Voting slightly lower (0.202), so overall early-rank performance is similar across all three strategies. Waterfall produced the strongest Precision@3 (0.139), indicating better short-list relevance in the top few results, while Confidence led at Precision@5 (0.133). At larger cutoffs, Confidence and Voting were nearly tied on Recall@10, with Waterfall slightly behind. In practice, these results suggest that orchestration variants are competitive but not dramatically separated, with Confidence showing the most balanced behavior and Waterfall favoring early precision at small k.

We evaluated all provided orchestration strategies: Waterfall, Voting, and Confidence. **Since Confidence achieved the best overall MRR on the full corpus, we selected it as the reference baseline for detailed answer synthesis, efficiency measurement, and failure analysis.** The other strategies remain relevant in the failure taxonomy and later adaptive recovery, where the system can switch strategy when the selected route appears unreliable.

In [13]:
# Save separate orchestration results
orch_summary_path = OUT_DIR / 'metrics_orchestration_summary.csv'
orch_runs_path = OUT_DIR / 'runs_orchestration.json'

orch_summary_df.to_csv(orch_summary_path, index=False)
orch_runs_path.write_text(json.dumps(orch_runs, indent=2), encoding='utf-8')

print('Saved:')
print('-', orch_summary_path)
print('-', orch_runs_path)


Saved:
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_orchestration_summary.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/runs_orchestration.json


## 1.8 Measure Baseline Performance + Answer Synthesis (Required)
This section provides the required benchmark reporting for the **reference baseline configuration** used in the rest of the project, including retrieval quality, answer quality, and efficiency.

**Reference baseline configuration**
- Scope: `full_corpus`
- Strategy: `Confidence` orchestration
- Candidate depth: `TOP_N = 100`
- Metrics source: same benchmark QA + qrels used above
- Answer synthesis mode: `llm` using the previous-semester sentence-selection plus `Mistral-7B-Instruct` procedure

**Previous-semester synthesis behavior.** In the earlier baseline, answer synthesis was not purely extractive: the system first selected a small set of query-relevant sentences from the retrieved documents and then passed that reduced context to `Mistral-7B-Instruct` to generate a short answer.

**Our reproduction in Step 1.** We now reproduce that same behavior directly in the baseline cell: select the best query-relevant sentences from the retrieved documents and pass them to `Mistral-7B-Instruct` for deterministic answer generation. The extractive path remains only as an exception fallback if LLM synthesis fails at runtime.

**Hardware note.** This reproduction is intended to run on a `CUDA`-capable `NVIDIA GPU`. In Google Colab, choose `Runtime` -> `Change runtime type` -> `GPU`. CPU-only runtimes may be too slow or may fail when loading `Mistral-7B-Instruct`.


In [14]:

# Required baseline report: retrieval quality + answer quality + efficiency
import re
import math
from IPython.display import HTML

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

REFERENCE_BASELINE = {
    'scope': EVAL_SCOPE,
    'strategy': 'Confidence',
    'top_n': TOP_N,
}

if 'orch_summary_df' not in globals():
    raise RuntimeError('Run orchestration evaluation section first to create orch_summary_df.')

ref_row = orch_summary_df.loc[orch_summary_df['method'] == REFERENCE_BASELINE['strategy']]
if ref_row.empty:
    raise RuntimeError('Reference strategy not found in orch_summary_df.')

# Retrieval quality (required)
retrieval_quality = ref_row[['method','queries_evaluated','MRR','Precision@1','Precision@3','Precision@5','Precision@10','Recall@1','Recall@3','Recall@5','Recall@10']].copy()

# ---- Answer synthesis (aligned to baseline Step 2 procedure) ----
# Procedure: top docs -> best query-relevant sentences -> [INST] prompt -> deterministic generation
# Set to True to run the same LLM-based synthesis as Step 2 baseline.
USE_STEP2_LLM_SYNTHESIS = True


def _tok(s: str):
    s = (s or '').lower()
    s = re.sub(r'[^\w\s]', ' ', s)
    return [t for t in s.split() if t]


def _token_f1(pred: str, gold: str):
    p = _tok(pred)
    g = _tok(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    from collections import Counter as _Counter
    pc, gc = _Counter(p), _Counter(g)
    overlap = sum(min(pc[t], gc[t]) for t in pc.keys() & gc.keys())
    if overlap == 0:
        return 0.0
    prec = overlap / len(p)
    rec = overlap / len(g)
    return 2 * prec * rec / (prec + rec)


def select_best_sentences(raw_text: str, query: str, max_sentences: int = 2):
    if not raw_text:
        return []
    q_terms = set(_tok(query))
    sents = re.split(r'(?<=[.!?])\s+', raw_text)
    scored = []
    for s in sents:
        s_clean = s.strip()
        if len(s_clean) < 20:
            continue
        s_terms = set(_tok(s_clean))
        overlap = len(q_terms & s_terms) / max(len(q_terms), 1)
        scored.append((overlap, s_clean))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [s for _, s in scored[:max_sentences]]


def _extractive_fallback_answer(query: str, docs):
    context_blocks = []
    for d in docs[:7]:
        raw = d.metadata.get('original_text') or d.page_content or ''
        best = select_best_sentences(raw, query, max_sentences=2)
        if best:
            context_blocks.append(' '.join(best))
    if not context_blocks:
        return 'NOT FOUND IN CONTEXT'
    all_sents = ' '.join(context_blocks)
    sents = re.split(r'(?<=[.!?])\s+', all_sents)
    return sents[0].strip() if sents and sents[0].strip() else 'NOT FOUND IN CONTEXT'


LLM_TOKENIZER = None
LLM_MODEL = None
LLM_READY = False


def _load_step2_llm(model_name='mistralai/Mistral-7B-Instruct-v0.2'):
    global LLM_TOKENIZER, LLM_MODEL, LLM_READY
    if LLM_READY:
        return
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    LLM_TOKENIZER = AutoTokenizer.from_pretrained(model_name)
    LLM_MODEL = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        torch_dtype=torch.float16,
    )
    LLM_READY = True


def _step2_style_llm_answer(query: str, docs, max_new_tokens: int = 128):
    import torch
    context_blocks = []
    for d in docs[:7]:
        raw = d.metadata.get('original_text') or d.page_content or ''
        best_sents = select_best_sentences(raw, query, max_sentences=2)
        if best_sents:
            context_blocks.append(' '.join(best_sents))
    context = '\n'.join(context_blocks)

    prompt = f"""[INST]
Answer the question using the context.
Give a short, factual answer.
If the answer is not clearly supported by the context, say "NOT FOUND IN CONTEXT".

Context:
{context}

Question: {query}
[/INST]
"""

    inputs = LLM_TOKENIZER(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
    ).to(LLM_MODEL.device)

    with torch.no_grad():
        outputs = LLM_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    decoded = LLM_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
    return decoded.split('[/INST]')[-1].strip()


def quick_mistral_answer(
    query: str,
    docs,
    sample_docs: int = 3,
    max_sentences_per_doc: int = 1,
    max_new_tokens: int = 48,
    max_input_length: int = 768,
):
    """Lightweight Mistral synthesis for quick notebook experiments."""
    if sample_docs <= 0:
        return 'NOT FOUND IN CONTEXT'
    _load_step2_llm()
    import torch

    context_blocks = []
    for d in docs[:sample_docs]:
        raw = d.metadata.get('original_text') or d.page_content or ''
        best_sents = select_best_sentences(raw, query, max_sentences=max_sentences_per_doc)
        if best_sents:
            context_blocks.append(' '.join(best_sents))

    if not context_blocks:
        return 'NOT FOUND IN CONTEXT'

    context = '\n'.join(context_blocks)
    prompt = f'''[INST]
Answer the question using only the context.
Be concise and factual.
If the answer is not clearly supported, say "NOT FOUND IN CONTEXT".

Context:
{context}

Question: {query}
[/INST]
'''

    inputs = LLM_TOKENIZER(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=max_input_length,
    ).to(LLM_MODEL.device)

    with torch.no_grad():
        outputs = LLM_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=LLM_TOKENIZER.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[-1]
    decoded = LLM_TOKENIZER.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return decoded.strip() or 'NOT FOUND IN CONTEXT'


def synthesize_answer(query: str, docs):
    if USE_STEP2_LLM_SYNTHESIS:
        try:
            _load_step2_llm()
            return _step2_style_llm_answer(query, docs)
        except Exception as e:
            print(f"[WARN] LLM synthesis unavailable, falling back to extractive: {e}")
    return _extractive_fallback_answer(query, docs)


# Gold answer accessor (robust to benchmark schema variants)
def _get_gold_answer(q_item: dict):
    for key in ['answer', 'gold_answer', 'reference_answer', 'expected_answer', 'answers']:
        if key in q_item and q_item[key] is not None:
            v = q_item[key]
            if isinstance(v, list):
                return ' '.join(str(x) for x in v if x is not None)
            return str(v)
    return ''


# Answer quality + efficiency + per-query table
conf_run = orch_runs[REFERENCE_BASELINE['strategy']]
qids = sorted(set(conf_run.keys()) & set(qrels.keys()))

qa_compare_rows = []
latencies = []
answerable_at_5 = []
first_rel_rank = []
f1_scores = []
exact_match = []

qa_by_id = {str(q['id']): q for q in qa_data}

for qid in tqdm(qids, desc='Answer synthesis eval (reference baseline)'):
    q_item = qa_by_id[qid]
    question = q_item['question']

    t0 = time.perf_counter()
    docs = orchestration_methods[REFERENCE_BASELINE['strategy']].search(question, top_k=TOP_N)
    answer = synthesize_answer(question, docs)
    dt = time.perf_counter() - t0
    latencies.append(dt)

    ranked_ids = normalize_docs(docs, top_n=TOP_N)
    rel = qrels.get(qid, set())

    top5_hit = any(d in rel for d in ranked_ids[:5])
    answerable_at_5.append(1 if top5_hit else 0)

    rr = reciprocal_rank(ranked_ids, rel)
    first_rel_rank.append((1.0 / rr) if rr > 0 else math.inf)

    gold = _get_gold_answer(q_item)
    f1 = _token_f1(answer, gold) if gold else np.nan
    em = int(answer.strip().lower() == gold.strip().lower()) if gold else np.nan
    if not np.isnan(f1):
        f1_scores.append(float(f1))
    if not np.isnan(em):
        exact_match.append(int(em))

    qa_compare_rows.append({
        'qid': qid,
        'question': question,
        'generated_answer': answer,
        'gold_answer': gold,
        'token_f1': f1,
        'exact_match': em,
        'answerable@5_proxy': bool(top5_hit),
        'latency_sec': dt,
    })

baseline_qa_compare_df = pd.DataFrame(qa_compare_rows)

answer_quality = pd.DataFrame([{
    'method': REFERENCE_BASELINE['strategy'],
    'synthesis_mode': ('step2_llm' if USE_STEP2_LLM_SYNTHESIS else 'extractive_fallback'),
    'Answerable@5 (proxy)': float(np.mean(answerable_at_5)) if answerable_at_5 else 0.0,
    'Median First Relevant Rank': float(np.median([r for r in first_rel_rank if np.isfinite(r)])) if any(np.isfinite(r) for r in first_rel_rank) else np.inf,
    'Token-F1 (vs gold)': float(np.mean(f1_scores)) if f1_scores else np.nan,
    'Exact-Match (vs gold)': float(np.mean(exact_match)) if exact_match else np.nan,
}])

efficiency = pd.DataFrame([{
    'method': REFERENCE_BASELINE['strategy'],
    'mean_latency_sec': float(np.mean(latencies)) if latencies else 0.0,
    'p95_latency_sec': float(np.quantile(latencies, 0.95)) if latencies else 0.0,
    'approx_cost_proxy': ('Medium/High (LLM decoding per query)' if USE_STEP2_LLM_SYNTHESIS else 'Low (extractive fallback)'),
}])

print('Reference baseline configuration:')
print(REFERENCE_BASELINE)
print('\nRetrieval quality (benchmark):')
display(retrieval_quality)
print('\nAnswer quality metrics:')
display(answer_quality)
print('\nSystem efficiency indicators:')
display(efficiency)
print('\nQuestion vs generated answer table (for manual comparison):')
qa_compare_full_cols = ['qid','question','generated_answer','gold_answer','token_f1','exact_match','answerable@5_proxy','latency_sec']
qa_compare_full = baseline_qa_compare_df[qa_compare_full_cols].copy()
display(HTML(qa_compare_full.to_html(index=False, escape=False)))

print("\nFull-text QA view (guaranteed no truncation):")
for _, row in qa_compare_full.iterrows():
    print("=" * 120)
    print(f"QID: {row['qid']}")
    print(f"QUESTION: {row['question']}")
    print(f"GENERATED_ANSWER: {row['generated_answer']}")
    print(f"GOLD_ANSWER: {row['gold_answer']}")
    print(f"token_f1={row['token_f1']} | exact_match={row['exact_match']} | answerable@5={row['answerable@5_proxy']} | latency={row['latency_sec']:.4f}s")

# Reusable artifacts in notebook runtime
baseline_required_metrics = {
    'reference': REFERENCE_BASELINE,
    'retrieval_quality': retrieval_quality,
    'answer_quality': answer_quality,
    'efficiency': efficiency,
    'qa_table': baseline_qa_compare_df,
}


Answer synthesis eval (reference baseline):   0%|          | 0/24 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...


System efficiency indicators:


,method,mean_latency_sec,p95_latency_sec,approx_cost_proxy
0,Confidence,8.17514,8.154286,Medium/High (LLM decoding per query)



Question vs generated answer table (for manual comparison):


qid,question,generated_answer,gold_answer,token_f1,exact_match,answerable@5_proxy,latency_sec
10,how would you make ferzlizer without carbon emissions?,"Decarbonizing fertilizer production can be achieved by identifying alternatives to synthetic nitrogen fertilizers that do not rely on carbon emissions. For instance, scientists have explored producing fertilizer using bacteria that convert carbon dioxide and water into climate-neutral chemicals like methanol (Büchel et al., 2022). This process, known as methanol synthesis, utilizes renewable energy and reduces the ecological footprint of the chemical industry. Additionally, the study ""Food security implications of transitioning synthetic nitrogen fertilizers to net-zero emissions"" suggests that decar","reducing carbon emissions in ferzlizer produczon requires shifting away from fossil fuel- based hydrogen and adopzng more sustainable alternazves. one method is electrificazon using renewable energy, where hydrogen is produced via water electrolysis powered by solar or wind energy, eliminazng direct emissions but requiring significantly more energy. a third option is hydrogen produczon from biomass, which can be carbon-neutral if using waste materials but competes with food produczon for land and water. addizonally, improving ferzlizer efficiency by reducing overuse and food waste can lower overall demand, while decentralized produczon in regions with abundant renewables or biomass can enhance food security. a combinazon of these strategies, tailored to local conditions, offers the most effeczve path to decarbonizing ferzlizer produczon, mitigating climate impact while maintaining agricultural productivity. this is about explaining a phenomenon based on eth research.",0.230415,0,False,70.625476
11,what research is eth famous for?,"ETH Zurich is famous for conducting research in various fields, including oceanic sink for man-made CO2, agricultural sciences with a focus on smart farming, personalized treatment methods for pancreatic cancer, and determining real estate bubbles in Switzerland, among others. Einstein, a famous alumnus, made significant contributions to physics.","as a research-intensive university, eth conducts research across a wide spectrum of disciplines. what are you interested in? as an official bot, it should not favor one department over another. when questions about being currently famous or successful arise, they need to be deflected.",0.148936,0,True,4.553707
12,how much of eth’s electricity consumpzon is due to compuzng? how did that develop over the years?,"The context does not provide information on how much of ETH Zurich's electricity consumption is due to computing specifically. However, it mentions that the university consumes approximately 112 gigawatt hours of electricity per year, excluding the electricity needs of the Swiss National Supercomputing Centre (CSCS). It also mentions that ETH Zurich has already bought a significant portion of its electricity for the next three years, and that the university aims to limit foreseeable cost increases by implementing electricity savings targets. There is no information on how electricity consumption for computing has developed over the years.\n\nNOT FO","we have very litle data on this. in 2017, compuzng at eth zurich produced 13,500 tons of co2, which is about 10% of the total emissions produced. eth zurich consumes about 40 gwh of electricity per year; this number has been increasing over the years. eth zurich has the goal of net-zero emissions by 2030. it is important to say, “we do not know,” but go out and find related informazon.",0.315789,0,False,8.147828
13,what are pseudocereals and who does research on them?,"Pseudocereals are plants whose seeds are used as grain-like foods, even though they are not true cereals. Buckwheat, quinoa, and amaranth are examples of pseudocereals. Hess, who studied environmental sciences at ETH Zurich, conducted research on duckweed but also worked with pseudocereals


Full-text QA view (guaranteed no truncation):
QID: 10
QUESTION: how would you make ferzlizer without carbon emissions?
GENERATED_ANSWER: Decarbonizing fertilizer production can be achieved by identifying alternatives to synthetic nitrogen fertilizers that do not rely on carbon emissions. For instance, scientists have explored producing fertilizer using bacteria that convert carbon dioxide and water into climate-neutral chemicals like methanol (Büchel et al., 2022). This process, known as methanol synthesis, utilizes renewable energy and reduces the ecological footprint of the chemical industry. Additionally, the study "Food security implications of transitioning synthetic nitrogen fertilizers to net-zero emissions" suggests that decar
GOLD_ANSWER: reducing carbon emissions in ferzlizer produczon requires shifting away from fossil fuel- based hydrogen and adopzng more sustainable alternazves. one method is electrificazon using renewable energy, where hydrogen is produced via water elec

### 1.9 Baseline system efficiency indicators
For the reference baseline (Confidence, 24 benchmark queries), retrieval quality was modest: MRR=0.209, Precision@5=0.133, and Recall@10=0.032, indicating that relevant evidence was often not ranked highly enough for answer synthesis. Answer quality was also limited: using the step2_llm Mistral synthesis path, Answerable@5 was 0.50, the median first relevant rank was 5, average token-F1 against the gold answers was 0.184, and exact match was 0.0 (which is probably too strict configured). System efficiency showed a mean latency of 76.16s per query and a p95 latency of 85.06s, with an approximate cost proxy of Medium/High because each query requires LLM decoding.


### 2. Qualitative Observations from Baseline Answer Synthesis

Manual inspection of the baseline answer synthesis outputs shows that the automatic metrics do not fully explain system behavior. Several examples reveal recurring patterns around retrieval quality, answer format, and metric limitations.

For the query **“who at ETH received ERC grants?”** (`QID 3`), the baseline retrieved answerable evidence in the top-5 window, but the generated answer focused on aggregate ERC funding statistics and mentioned only a few researchers. The gold answer, however, expects a list of specific ERC grant recipients. This shows that `answerable@5=True` is not sufficient for answer correctness: even when relevant material is retrieved, the synthesizer may select the wrong level of detail or answer format.

For **“when did the InSight get to Mars?”** (`QID 4`), the generated answer states that InSight reached Mars on **November 26, 2018**, which matches the gold answer **“26 november 2018”**. However, exact match is still zero because the generated answer is a longer sentence and uses a different date surface form. This illustrates that exact match is too strict for some questions and can underestimate correctness when the right fact is present in a different wording.

For **“what did Prof. Schubert say about flying?”** (`QID 5`), the answer discusses flight costs and compares airplanes to railways, but it misses the gold answer’s central claim that flying is too cheap and that surcharges on air fares are a step in the right direction. Since `answerable@5=False`, this appears to be caused by missing or incomplete evidence in the answer window, followed by generation from only partially related context.

For **“what is e-sling?”** (`QID 6`), the system returns `NOT FOUND IN CONTEXT`, while the gold answer identifies e-Sling as a four-seat electric airplane built by ETH students. This is a useful example of a wrong abstention: the system behaves conservatively, but only because the relevant evidence was not retrieved into the top-5 answer window.

For **“who are famous ETH alumni?”** (`QID 7`), the generated answer lists ETH-related Olympic athletes and an alumni association figure, while the gold answer expects well-known historical ETH alumni such as Albert Einstein, Wilhelm Conrad Röntgen, Felix Bloch, and others. This is a clear topic-drift case: the answer is fluent and ETH-related, but it addresses the wrong interpretation of “alumni”.

Overall, these examples show three important limitations of the baseline. First, retrieval quality strongly affects answer quality: when the relevant evidence is absent from the top answer window, the generator often answers from adjacent but incorrect context. Second, even when answerable evidence is present, the synthesizer may choose the wrong granularity or format. Third, exact match is a strict lower-bound metric and should be complemented with token-F1 and qualitative inspection, especially for short factual answers such as dates.

## 2.2. Analyze Failure Cases with Structured Taxonomy (Required)
The taxonomy below classifies representative failure cases for the same reference baseline (`Confidence`, `full_corpus`). It is a heuristic diagnostic pass: labels are useful for finding patterns, but representative examples should still be inspected manually before being used in the report.

**Taxonomy definitions used in this notebook run**
- `retrieval_failure`: no relevant evidence retrieved in top-`K_retrieval`
- `ranking_failure`: relevant evidence exists in top-`K_retrieval` but not in top-`K_answer`
- `synthesis_failure`: relevant evidence exists in top-`K_answer`, but the generated answer has low overlap with the gold answer and does not contain an equivalent date/fact proxy
- `grounding_failure`: generated answer has weak lexical support in the top evidence window (proxy)
- `ambiguity_failure`: query appears under-specified and the system still returns an answerable result
- `contradiction_failure`: top evidence has a temporal mismatch with the query year (proxy, manually inspect before using)
- `orchestration_failure`: Confidence fails while another strategy succeeds on same query
- `overconfidence_failure`: system gives a non-abstaining answer despite no relevant evidence in top-`K_retrieval`


In [16]:
# Structured failure taxonomy with representative examples
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
import re
from collections import Counter
from IPython.display import HTML

K_RETRIEVAL = 20
K_ANSWER = 5
SYNTHESIS_F1_THRESHOLD = 0.35
GROUNDING_SUPPORT_THRESHOLD = 0.25

# Top-20 is the retrieval window; top-5 is the answer window used by synthesis.
def _tokenize(s: str):
    s = (s or '').lower()
    s = re.sub(r'[^\w\s]', ' ', s)
    return [t for t in s.split() if t]

def _token_f1_proxy(pred: str, gold: str):
    p = _tokenize(pred)
    g = _tokenize(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    overlap = sum(min(pc[t], gc[t]) for t in pc.keys() & gc.keys())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def _lex_overlap_ratio(query: str, text: str):
    q = set(_tokenize(query))
    d = set(_tokenize(text))
    if not q:
        return 0.0
    return len(q & d) / len(q)

def _answer_support_ratio(answer: str, evidence: str):
    answer_terms = set(_tokenize(answer))
    evidence_terms = set(_tokenize(evidence))
    if not answer_terms:
        return 0.0
    return len(answer_terms & evidence_terms) / len(answer_terms)

def _is_non_abstaining_answer(answer: str):
    a = (answer or '').strip().lower()
    return bool(a) and 'not found in context' not in a

def _is_ambiguous_query(q: str):
    toks = _tokenize(q)
    pronouns = {'it','they','this','that','these','those','he','she','them'}
    pronoun_count = sum(t in pronouns for t in toks)
    # Pronouns are treated as ambiguity signals only when little other context is present.
    return (len(toks) <= 4) or (pronoun_count > 0 and len(toks) <= 8)

def _extract_date_keys(text: str):
    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
    }
    t = (text or '').lower()
    dates = set()
    for day, month, year in re.findall(r'\b(\d{1,2})\s+(' + '|'.join(months) + r')\s*,?\s*(\d{4})\b', t):
        dates.add(f'{year}-{months[month]}-{int(day):02d}')
    for month, day, year in re.findall(r'\b(' + '|'.join(months) + r')\s+(\d{1,2}),?\s*(\d{4})\b', t):
        dates.add(f'{year}-{months[month]}-{int(day):02d}')
    return dates

def _contains_gold_proxy(answer: str, gold: str):
    answer_norm = ' '.join(_tokenize(answer))
    gold_norm = ' '.join(_tokenize(gold))
    if gold_norm and gold_norm in answer_norm:
        return True
    answer_dates = _extract_date_keys(answer)
    gold_dates = _extract_date_keys(gold)
    return bool(answer_dates & gold_dates)

def _has_temporal_contradiction(query: str, docs):
    query_years = set(re.findall(r'\b(19\d{2}|20\d{2})\b', query or ''))
    if not query_years:
        return False
    evidence_years = set()
    for d in docs[:K_ANSWER]:
        txt = (d.metadata.get('original_text') or d.page_content or '')[:600]
        evidence_years.update(re.findall(r'\b(19\d{2}|20\d{2})\b', txt))
    # This is a conservative temporal-mismatch proxy, not semantic contradiction detection.
    return bool(evidence_years) and not bool(query_years & evidence_years) and len(evidence_years) >= 2

ref = orchestration_methods['Confidence']
alt_names = [k for k in orchestration_methods.keys() if k != 'Confidence']

# Pull generated answers from section 1.8 if available
qa_answer_map = {}
qa_gold_map = {}
if 'baseline_qa_compare_df' in globals() and isinstance(baseline_qa_compare_df, pd.DataFrame):
    if 'qid' in baseline_qa_compare_df.columns:
        for _, r in baseline_qa_compare_df.iterrows():
            qid = str(r.get('qid'))
            qa_answer_map[qid] = r.get('generated_answer', '')
            qa_gold_map[qid] = r.get('gold_answer', '')

rows = []
for q in tqdm(qa_data, desc='Failure taxonomy scan'):
    qid = str(q['id'])
    query = q['question']
    relevant = qrels.get(qid, set())

    ref_docs = ref.search(query, top_k=TOP_N)
    ref_ids = normalize_docs(ref_docs, top_n=TOP_N)

    has_rel_topk = any(d in relevant for d in ref_ids[:K_RETRIEVAL])
    has_rel_topa = any(d in relevant for d in ref_ids[:K_ANSWER])
    answer = qa_answer_map.get(qid, '')
    gold = qa_gold_map.get(qid, '')
    non_abstaining = _is_non_abstaining_answer(answer)
    answer_f1 = _token_f1_proxy(answer, gold) if answer and gold else None
    contains_gold_proxy = _contains_gold_proxy(answer, gold) if answer and gold else False

    # answerability proxy: at least one relevant chunk in top answer window
    answered_ok = has_rel_topa

    # Compare with alternate strategy success to flag orchestration miss
    alt_succeeds = False
    alt_used = None
    for an in alt_names:
        a_docs = orchestration_methods[an].search(query, top_k=TOP_N)
        a_ids = normalize_docs(a_docs, top_n=TOP_N)
        if any(d in relevant for d in a_ids[:K_ANSWER]):
            alt_succeeds = True
            alt_used = an
            break

    top_text = ''
    if ref_docs:
        top_text = ref_docs[0].metadata.get('original_text') or ref_docs[0].page_content or ''
    overlap = _lex_overlap_ratio(query, top_text)
    win_text = ' '.join((d.metadata.get('original_text') or d.page_content or '')[:500] for d in ref_docs[:K_ANSWER])

    labels = []
    label_reasons = []
    if not has_rel_topk:
        labels.append('retrieval_failure')
        label_reasons.append(f'no relevant doc in top-{K_RETRIEVAL}')
    if has_rel_topk and not has_rel_topa:
        labels.append('ranking_failure')
        label_reasons.append(f'relevant doc found in top-{K_RETRIEVAL}, but not top-{K_ANSWER}')
    if has_rel_topa and answer and gold and (answer_f1 is not None) and answer_f1 < SYNTHESIS_F1_THRESHOLD and not contains_gold_proxy:
        labels.append('synthesis_failure')
        label_reasons.append(f'top-{K_ANSWER} evidence exists, but answer/gold F1={answer_f1:.2f}')

    if non_abstaining and win_text:
        support = _answer_support_ratio(answer, win_text)
        if support < GROUNDING_SUPPORT_THRESHOLD:
            labels.append('grounding_failure')
            label_reasons.append(f'answer/evidence lexical support={support:.2f}')
    else:
        support = None

    if _is_ambiguous_query(query) and answered_ok and non_abstaining:
        labels.append('ambiguity_failure')
        label_reasons.append('short or context-dependent query answered directly')

    if _has_temporal_contradiction(query, ref_docs):
        labels.append('contradiction_failure')
        label_reasons.append('query year not found; top evidence contains other years')

    if (not answered_ok) and alt_succeeds:
        labels.append('orchestration_failure')
        label_reasons.append(f'Confidence misses top-{K_ANSWER}; {alt_used} succeeds')

    if (not has_rel_topk) and non_abstaining and overlap >= 0.5:
        labels.append('overconfidence_failure')
        label_reasons.append(f'non-abstaining answer despite no top-{K_RETRIEVAL} evidence; top-1 query overlap={overlap:.2f}')

    rows.append({
        'qid': qid,
        'query': query,
        'generated_answer': answer,
        'gold_answer': gold,
        'labels': labels,
        'label_reasons': label_reasons,
        'reference_success_top5': bool(answered_ok),
        'alt_success_top5': bool(alt_succeeds),
        'alt_strategy': alt_used,
        'query_overlap_top1': float(overlap),
        'answer_gold_f1': answer_f1,
        'answer_evidence_support': support,
    })

failure_df = pd.DataFrame(rows)

# Expand and aggregate
expanded = []
for r in rows:
    if not r['labels']:
        continue
    for lab, reason in zip(r['labels'], r['label_reasons']):
        expanded.append({
            'failure_type': lab,
            'qid': r['qid'],
            'query': r['query'],
            'generated_answer': r.get('generated_answer', ''),
            'gold_answer': r.get('gold_answer', ''),
            'reason': reason,
            'alt_strategy': r['alt_strategy'],
        })

fail_expanded_df = pd.DataFrame(expanded)
if fail_expanded_df.empty:
    taxonomy_summary = pd.DataFrame(columns=['failure_type', 'count'])
else:
    taxonomy_summary = fail_expanded_df.groupby('failure_type', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False)

print('Structured failure taxonomy (reference baseline):')
display(taxonomy_summary)

print('Representative examples (up to 2 per failure type; manually inspect before report use):')
example_rows = []
for ft in [
    'retrieval_failure','ranking_failure','synthesis_failure','grounding_failure',
    'ambiguity_failure','contradiction_failure','orchestration_failure','overconfidence_failure'
]:
    if fail_expanded_df.empty:
        continue
    sub = fail_expanded_df[fail_expanded_df['failure_type']==ft].copy()
    if 'generated_answer' in sub.columns and 'gold_answer' in sub.columns:
        sub['_has_answer'] = sub['generated_answer'].fillna('').astype(str).str.len() > 0
        sub['_has_gold'] = sub['gold_answer'].fillna('').astype(str).str.len() > 0
        # Representative examples should be interpretable in the report; keep raw counts above unchanged.
        report_ready = sub[sub['_has_answer'] & sub['_has_gold']]
        if not report_ready.empty:
            sub = report_ready
        sub = sub.sort_values(['_has_answer', '_has_gold'], ascending=False)
    sub = sub.head(2)
    for _, rr in sub.iterrows():
        example_rows.append({
            'failure_type': ft,
            'qid': rr['qid'],
            'query': rr['query'],
            'generated_answer': rr.get('generated_answer', ''),
            'gold_answer': rr.get('gold_answer', ''),
            'reason': rr.get('reason', ''),
            'alt_strategy_if_any': rr['alt_strategy'],
        })

taxonomy_examples_df = pd.DataFrame(example_rows)
if not taxonomy_examples_df.empty:
    ex_cols = ['failure_type','qid','query','generated_answer','gold_answer','reason','alt_strategy_if_any']
    display(HTML(taxonomy_examples_df[ex_cols].to_html(index=False, escape=False)))
else:
    display(taxonomy_examples_df)

# Save in-memory artifacts for downstream steps
baseline_failure_taxonomy = {
    'taxonomy_summary': taxonomy_summary,
    'taxonomy_examples': taxonomy_examples_df,
    'per_query_labels': failure_df,
}

print("\nFull-text failure examples (guaranteed no truncation):")
if taxonomy_examples_df.empty:
    print("No failure examples found in this run.")
else:
    for _, row in taxonomy_examples_df.iterrows():
        print("-" * 120)
        print(f"failure_type: {row['failure_type']} | qid: {row['qid']} | alt_strategy: {row['alt_strategy_if_any']}")
        print(f"reason: {row['reason']}")
        print(f"QUESTION: {row['query']}")
        print(f"GENERATED_ANSWER: {row['generated_answer']}")
        print(f"GOLD_ANSWER: {row['gold_answer']}")


Failure taxonomy scan:   0%|          | 0/25 [00:00<?, ?it/s]

Structured failure taxonomy (reference baseline):


,failure_type,count
5,synthesis_failure,10
4,retrieval_failure,8
3,ranking_failure,5
2,overconfidence_failure,4
1,orchestration_failure,1
0,contradiction_failure,1


Representative examples (up to 2 per failure type; manually inspect before report use):


failure_type,qid,query,generated_answer,gold_answer,reason,alt_strategy_if_any
retrieval_failure,2,who were the rectors of eth between 2017 and 2022?,NOT FOUND IN CONTEXT. The context does not provide information about the rectors of ETH Zurich between 2017 and 2022.,"sarah springman, günther dissertori.",no relevant doc in top-20,None
retrieval_failure,6,what is e-sling?,"NOT FOUND IN CONTEXT. The context does not mention anything about an ""e-sling"".","4-seated electric airplane, built by 20 electrical and mechanical engineering students at eth zurich.",no relevant doc in top-20,None
ranking_failure,5,what did prof. schubert say about flying?,"Prof. Schubert mentioned that blog kosten (costs of blogging) for flying are 60% higher than the original costs, and if ticket charges are taken into account, the factor would be even greater. She also stated that planes today do what railways did in the mid-19th century.","flying is too cheap. if we want to reduce flying, surcharges on air fares are certainly a step in the right direction.","relevant doc found in top-20, but not top-5",None
ranking_failure,12,how much of eth’s electricity consumpzon is due to compuzng? how did that develop over the years?,"The context does not provide information on how much of ETH Zurich's electricity consumption is due to computing specifically. However, it mentions that the university consumes approximately 112 gigawatt hours of electricity per year, excluding the electricity needs of the Swiss National Supercomputing Centre (CSCS). It also mentions that ETH Zurich has already bought a significant portion of its electricity for the next three years, and that the university aims to limit foreseeable cost increases by implementing electricity savings targets. There is no information on how electricity consumption for computing has developed over the years.\n\nNOT FO","we have very litle data on this. in 2017, compuzng at eth zurich produced 13,500 tons of co2, which is about 10% of the total emissions produced. eth zurich consumes about 40 gwh of electricity per year; this number has been increasing over the years. eth zurich has the goal of net-zero emissions by 2030. it is important to say, “we do not know,” but go out and find related informazon.","relevant doc found in top-20, but not top-5",None
synthesis_failure,3,who at eth received erc grants?,"More than 80 researchers at ETH Zurich have received ERC grants, including two researchers named Detlef Günther and Vorholt who received advanced grants. Eth researchers have been applying for EU funding through ERC grants since 2007 and have secured a total of 134 grants, worth around 242.1 million euros. Additionally, Eth Zurich researchers have received around 114 million Swiss francs from ERC advanced grants alone.","european research council grants: tobias donner, eliot ash, ursula keller, klaus ensslin, yiwen chu, judit szuláovi, sebaszno cantalupo, veerle sterken, rachel granoe, paolo crivelli, chriszan degen, jonathan home, lavinia heisenberg, tilman esslinger, ....","top-5 evidence exists, but answer/gold F1=0.02",Waterfall
synthesis_failure,8,who at eth currently works on research regarding climate change?,"Several professors at ETH Zurich currently work on research related to climate change. These include Professors Eleni Chatzi (climate physics), Reto Knutti (climate physics), Jing Wang (aspects of forest dynamics), Marloes Maathuis (mathematical and interdisciplinary research), Guillaume Habert (sustainable construction), and Helga (impact of early human activity on ecosystems of previously uninhabited eastern pacific islands). Additionally, Dr. Jake Alexander and Dr. Martin Frimmer are adjunct professors at ETH Zurich,","eth department of environmental systems science, d-usys, on the research side, anthony pat (climate policy) and several researchers (check); insztute for atmospheric and climate science; chair of hydrology and water resources development, d-baug, nadel on sustainable dev


Full-text failure examples (guaranteed no truncation):
------------------------------------------------------------------------------------------------------------------------
failure_type: retrieval_failure | qid: 2 | alt_strategy: None
reason: no relevant doc in top-20
QUESTION: who were the rectors of eth between 2017 and 2022?
GENERATED_ANSWER: NOT FOUND IN CONTEXT. The context does not provide information about the rectors of ETH Zurich between 2017 and 2022.
GOLD_ANSWER: sarah springman, günther dissertori.
------------------------------------------------------------------------------------------------------------------------
failure_type: retrieval_failure | qid: 6 | alt_strategy: None
reason: no relevant doc in top-20
QUESTION: what is e-sling?
GENERATED_ANSWER: NOT FOUND IN CONTEXT. The context does not mention anything about an "e-sling".
GOLD_ANSWER: 4-seated electric airplane, built by 20 electrical and mechanical engineering students at eth zurich.
-----------------------

### 2.3. Observations from the Structured Failure Taxonomy and Motivation for Proposed Extensions

|index|failure\_type|count|
|---|---|---|
|5|synthesis\_failure|10|
|4|retrieval\_failure|8|
|3|ranking\_failure|5|
|2|overconfidence\_failure|4|
|1|orchestration\_failure|1|
|0|contradiction\_failure|1|

The structured taxonomy identifies the main weaknesses of the reference baseline and motivates the reliability mechanisms implemented in Steps 2-4. The taxonomy is heuristic: it uses retrieval position, answer/gold overlap, and lightweight proxy signals rather than full semantic judgment. Therefore, the failure categories should be interpreted as diagnostic patterns rather than absolute labels.

The largest category is `synthesis_failure` with 10 cases under our heuristic. In these cases, relevant evidence was present in the top-5 answer window, but the generated answer still had weak lexical overlap with the gold answer. This suggests that retrieval success alone is not sufficient: the synthesizer may still choose the wrong level of detail, produce the wrong answer format, or focus on related but non-target facts. This directly motivates the `GroundednessAgent` and `CriticAgent` in Step 3, which check whether an answer is sufficiently supported and whether it should be revised or rejected.

The second largest category is `retrieval_failure` with 8 cases. Here, no relevant document was found in the top-20 retrieval window. This can lead either to explicit abstention, such as `NOT FOUND IN CONTEXT`, or to generation from semantically related but incorrect evidence. This motivates the `EvidenceSufficiencyAgent`, which estimates whether retrieved evidence is strong enough to answer, and the `AbstentionAgent`, which prevents the system from answering when evidence is insufficient.

`ranking_failure` occurs when relevant evidence is retrieved somewhere in the top-20 but does not reach the top-5 answer window used for synthesis. This shows that retrieval alone is not enough; the system also needs to reason about whether the evidence selected for answer generation is adequate. In Steps 2-4, this motivates the `RecoveryAgent`, which can retry with a different strategy or switch orchestration behavior when the initial evidence window is unreliable.

The `overconfidence_failure` examples are especially important for reliability. In these cases, the system produces fluent, non-abstaining answers despite having no relevant evidence in the top-20. The answers look plausible because the retrieved passages have lexical overlap with the query, but they are factually misaligned with the gold answer. This motivates the `TrustAgent`, which combines reliability signals into a trust score, and the `AbstentionAgent`, which blocks answers when trust is too low.

The single `orchestration_failure` case shows that the selected `Confidence` strategy can fail while another strategy, `Waterfall`, succeeds according to the top-5 evidence proxy. This supports the adaptive part of the proposed system: the `RecoveryAgent` can switch strategy when the initial route appears unreliable, rather than treating the first orchestration choice as final.

The `contradiction_failure` category should be interpreted cautiously. In Step 1, it is based on a lightweight temporal mismatch proxy rather than full semantic contradiction detection. We therefore use it mainly as motivation for including a `ContradictionAgent` in Step 3, where conflicting evidence is treated as a reliability signal that can reduce trust or trigger more cautious behavior.

Overall, the taxonomy supports the design direction taken in Steps 2-4. The baseline needs mechanisms that can detect weak evidence, verify support, reduce overconfident answering, abstain when necessary, and adapt retrieval or orchestration when the first attempt is unreliable. These are exactly the roles of the sufficiency, groundedness, contradiction, trust, abstention, critique, and recovery components in the reliable adaptive RAG system.